# Supervised fine-tuning for LFM2.5-8B-A1B with Halo

This notebook fine-tunes [LFM2.5-8B-A1B](https://huggingface.co/LiquidAI/LFM2.5-8B-A1B) on [UltraChat 200K](https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k) using [Halo](https://whitecircle.com/research/halo). The [Halo repository](https://github.com/whitecircle/halo) contains the source code.

Expert parallelism assigns different mixture-of-experts (MoE) experts to different GPUs. Halo sends each token to the GPU that owns its selected expert, so each GPU stores only part of the expert weights.

This two-way expert-parallel recipe distributes the model's 32 routed experts across two GPUs. Each process owns 16 experts. Halo gathers the weights during save and writes a standard Hugging Face checkpoint.


## Prerequisites

- Two matching NVIDIA H100, H200, B200, or B300 GPUs
- Docker with the NVIDIA Container Toolkit
- A large writable volume for model, dataset, and checkpoint files

This notebook does not run in Google Colab. Use a compatible two-GPU machine from your cloud provider or your own cluster. Allow several hours for a full UltraChat epoch on two B300 GPUs. This is a planning estimate, not a benchmark. After the first 20 steps, use the logged step time and total step count to refine the estimate.

## Recipe at a glance

This recipe uses full-parameter BF16 training with two-way expert parallelism.

| Setting | Value |
| --- | --- |
| GPUs | 2 |
| Expert parallel size | 2 |
| Dataset | UltraChat 200K, supervised split |
| Sequence length | 8,192 |
| Effective batch size | 16 |
| Output | Gathered Hugging Face checkpoint |

Do not enable context parallelism. LFM2 short-convolution layers operate across the sequence axis.


## Start the Halo notebook server

This notebook uses the public Halo image that matches the detected GPUs. Define these host paths before you start the container:

- `HALO_DIR`: an optional Halo clone for source reference
- `COOKBOOK_DIR`: a clone of this cookbook repository
- `DATA_DIR`: a large persistent volume for downloads and checkpoints

~~~bash
export HALO_DIR=/path/to/halo
export COOKBOOK_DIR=/path/to/cookbook
export DATA_DIR=/path/to/large/volume

git clone --recurse-submodules https://github.com/whitecircle/halo.git "$HALO_DIR"
git clone https://github.com/Liquid4All/cookbook.git "$COOKBOOK_DIR"

GPU_NAME="$(nvidia-smi --query-gpu=name --format=csv,noheader | head -n 1)"
case "$GPU_NAME" in
  *H100*|*H200*) HALO_IMAGE=public.ecr.aws/whitecircle/halo:hopper ;;
  *B200*|*B300*|*GB200*|*GB300*) HALO_IMAGE=public.ecr.aws/whitecircle/halo:blackwell ;;
  *) echo "Unsupported GPU: $GPU_NAME"; exit 1 ;;
esac
export HALO_IMAGE

docker pull "$HALO_IMAGE"
docker run --rm -it --gpus '"device=0,1"' \
  --ipc=host --shm-size=128g \
  --ulimit memlock=-1 --ulimit stack=67108864 \
  -p 8888:8888 \
  -e HF_HOME=/mnt/hf \
  -e HF_DATASETS_CACHE=/mnt/hf/datasets \
  -e TMPDIR=/mnt/tmp \
  -e HALO_DATA_ROOT=/mnt \
  -e CUDA_DEVICE_MAX_CONNECTIONS=1 \
  -v "$COOKBOOK_DIR":/cookbook \
  -v "$DATA_DIR":/mnt \
  -w /workspace \
  "$HALO_IMAGE" \
  jupyter lab --ip=0.0.0.0 --port=8888 --no-browser \
    --allow-root --notebook-dir=/cookbook/finetuning/notebooks
~~~

The container runs the Halo version packaged in the image. The `HALO_DIR` clone is for source reference and is not mounted over that package. `-p 8888:8888` publishes only the Jupyter port. `CUDA_DEVICE_MAX_CONNECTIONS=1` supports the DeepEP communication schedule used by this two-GPU MoE run. The single-GPU vision notebook does not need this setting.

Open the URL printed by Jupyter. Then open this notebook.


## Check the runtime

The notebook does not create a CUDA context before the distributed launch.


In [1]:
import subprocess
from pathlib import Path

RUN_ROOT = Path("/mnt/checkpoints")
CONFIG_PATH = RUN_ROOT / "lfm25-8b-a1b-ultrachat-ep2.yaml"
OUTPUT_PATH = RUN_ROOT / "lfm25-8b-a1b-ultrachat-ep2"

gpu_names = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=name",
        "--format=csv,noheader",
    ],
    text=True,
).strip().splitlines()
assert len(gpu_names) == 2, f"Expected 2 visible GPUs, found {len(gpu_names)}"

RUN_ROOT.mkdir(parents=True, exist_ok=True)
print(gpu_names)


['NVIDIA B300 SXM6 AC', 'NVIDIA B300 SXM6 AC']


## Data prep

The [UltraChat 200K](https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k) `train_sft` split already stores each conversation in a `messages` column. Halo downloads the split and applies the model's chat template. No separate conversion step is necessary. The configuration reserves 1% of the split for evaluation.


## Create the training configuration

This configuration follows Halo's LFM2 MoE recipe. The MoE settings have these purposes:

- `moe_balancing: bias_update` adjusts expert-selection biases during training because LFM2 does not use a router auxiliary loss.
- `save_sharded_ep: false` gathers expert weights and saves one standard Hugging Face checkpoint.
- `use_grouped_gemm: true` combines expert projections to reduce small kernel launches.
- `fp32_router: true` keeps routing calculations in FP32 for stable expert selection.
- `fp32_experts: false` keeps expert computation in BF16 to reduce memory use and training time.


In [ ]:
CONFIG_PATH.write_text(
    r"""
model_name_or_path: LiquidAI/LFM2.5-8B-A1B
moe_balancing: bias_update

dataset:
- HuggingFaceH4/ultrachat_200k@train_sft
conversation_field: messages
test_size: 0.01
train_on_completions_only: true
assistant_message_template: "<|im_start|>assistant\n"
pad_token: "<|pad|>"
eos_token: "<|im_end|>"

expert_parallel_size: 2
save_sharded_ep: false
use_grouped_gemm: true
fp32_router: true
fp32_experts: false

attn_implementation: flash_attention_2
use_liger_kernel: false
packing: true
max_length: 8192
bf16: true

per_device_train_batch_size: 1
per_device_eval_batch_size: 1
gradient_accumulation_steps: 8
num_train_epochs: 1.0
gradient_checkpointing: true
gradient_checkpointing_kwargs:
  use_reentrant: false

optim: adamw_torch_fused
learning_rate: 5.0e-06
lr_scheduler_type: cosine
warmup_steps: 32
max_grad_norm: 1.0

save_strategy: steps
save_steps: 1000
eval_strategy: steps
eval_steps: 300
save_total_limit: 1
save_only_model: true
output_dir: /mnt/checkpoints/lfm25-8b-a1b-ultrachat-ep2

logging_steps: 1
logging_first_step: true
report_to: none
remove_unused_columns: false
dataloader_num_workers: 2

use_peft: false
""".lstrip()
)
print(CONFIG_PATH.read_text())


## Check the launch command

The dry run prints the two-process command without loading the model.


In [ ]:
!halo launch sft {CONFIG_PATH} -n 2 --dry-run


## Launch training

This cell starts the full UltraChat run on both visible GPUs.


In [ ]:
!halo launch sft {CONFIG_PATH} -n 2


## Inference

The saved checkpoint uses the native Transformers format. This prompt checks that the checkpoint loads and follows a specific instruction. It is not a quality benchmark. To measure training gain, compare the base and fine-tuned checkpoints on the same held-out UltraChat examples with the same decoding settings.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(OUTPUT_PATH)
model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_PATH,
    dtype=torch.bfloat16,
    device_map="auto",
)

messages = [
    {
        "role": "user",
        "content": (
            "Explain how a heat pump warms a home in winter. "
            "Use three concise bullet points and avoid jargon."
        ),
    }
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)
output = model.generate(
    inputs,
    max_new_tokens=256,
    do_sample=False,
)
reply = tokenizer.decode(
    output[0, inputs.shape[-1] :],
    skip_special_tokens=True,
)
print(reply)


## Cleanup

Shut down Jupyter, then press `Ctrl+C` in the terminal that runs the container. The `--rm` option removes the stopped container. The files under `DATA_DIR` remain available. Stop or terminate the cloud GPU machine when you finish to avoid further charges.
